# Snowflake Intelligence Lab Guide

## What We Built So Far

In the **DT Lab Guide**, you built five silver Dynamic Iceberg Tables that aggregate bronze balloon game events into leaderboards, color stats, and real-time scores — all in open Iceberg format.

## What We'll Build Next

This notebook makes your silver data **AI-ready** by:

- Creating a **Semantic View** over the five silver tables (with Cortex Code or manually)
- Setting up an optional **Email tool** for the agent
- Configuring a **Snowflake Intelligence agent** that answers natural-language questions grounded in your data

**Prerequisites:**
- Silver Dynamic Iceberg Tables created and refreshed (from the DT Lab Guide)
- ACCOUNTADMIN role (for creating integrations and semantic views)
- A running warehouse

---

## Step 1: Configure Your Environment

Set the variables below to match your environment. All subsequent SQL cells reference these via Jinja templating.

> ### STOP — Update the variables below before proceeding!
>
> Make sure the values match your DT Lab Guide setup, then **run the cell below**.

In [ ]:
WAREHOUSE = 'BALLOON_WH'
DB_NAME = 'balloon_silver'
SCHEMA_NAME = 'silver'
SEMANTIC_VIEW_NAME = 'balloon_game_semantic_view'
AGENT_NAME = 'balloon_game_agent'

---

## Step 2: Set Context

In [ ]:
%%sql -r use_role
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r use_wh
USE WAREHOUSE {{WAREHOUSE}};

In [ ]:
%%sql -r use_schema
USE DATABASE {{DB_NAME}};
USE SCHEMA {{SCHEMA_NAME}};

---

## Step 3: Create Semantic View

The SQL cell below creates a **Semantic View** over your five silver tables. It defines:
- **Tables** — business-friendly aliases, synonyms, primary keys, and unique constraints
- **Relationships** — how tables join (player → leaderboard, color → color_stats)
- **Facts** — raw numeric columns (score, pops, bonus hits) with natural-language synonyms
- **Dimensions** — categorical/temporal columns (player name, balloon color, time windows)
- **Metrics** — pre-defined aggregations (total score, avg points per pop, player count, etc.)
- **AI SQL Generation** — context hints that help Cortex Analyst generate accurate queries

The cell uses Jinja variables (`{{DB_NAME}}`, `{{SCHEMA_NAME}}`, `{{SEMANTIC_VIEW_NAME}}`) from **Step 1**.

> **Want to regenerate?** You can also use **Cortex Code** (Cmd+K / Ctrl+K) to regenerate or customize the Semantic View. Use this prompt:
>
> *I run a balloon popping game and I want anyone on my team to ask questions in plain English. My data lives in 5 silver tables — a leaderboard, color breakdown, live scores in 15-second windows, a detailed player+color+window view, and color performance trends. Create a semantic view with facts, dimensions, metrics, and AI SQL generation hints that helps an AI answer any question about players, scores, colors, or trends.*

Run the cell below to create the Semantic View.

In [ ]:
%%sql -r coco_semantic_view
CREATE OR REPLACE SEMANTIC VIEW {{DB_NAME}}.{{SCHEMA_NAME}}.{{SEMANTIC_VIEW_NAME}}

  TABLES (
    player_leaderboard AS {{DB_NAME}}.{{SCHEMA_NAME}}.dt_player_leaderboard
      PRIMARY KEY (player)
      WITH SYNONYMS ('leaderboard', 'rankings', 'top players')
      COMMENT = 'Aggregated player scores: total score, bonus pops, last event timestamp per player',

    color_stats AS {{DB_NAME}}.{{SCHEMA_NAME}}.dt_balloon_color_stats
      UNIQUE (player, balloon_color)
      WITH SYNONYMS ('color breakdown', 'player colors', 'color scores')
      COMMENT = 'Per-player breakdown by balloon color: pops, points, and bonus hits',

    realtime_scores AS {{DB_NAME}}.{{SCHEMA_NAME}}.dt_realtime_scores
      WITH SYNONYMS ('live scores', 'recent scores', 'hot streaks')
      COMMENT = '15-second windowed score totals per player for time-series analysis',

    colored_pops AS {{DB_NAME}}.{{SCHEMA_NAME}}.dt_balloon_colored_pops
      WITH SYNONYMS ('detailed pops', 'player color windows')
      COMMENT = 'Most granular view: per-player, per-color pops in 15-second time windows',

    color_trends AS {{DB_NAME}}.{{SCHEMA_NAME}}.dt_color_performance_trends
      WITH SYNONYMS ('color trends', 'color performance', 'best colors')
      COMMENT = 'Average score per pop and total pops by balloon color over 15-second windows'
  )

  RELATIONSHIPS (
    color_stats_to_leaderboard AS
      color_stats (player) REFERENCES player_leaderboard,
    realtime_to_leaderboard AS
      realtime_scores (player) REFERENCES player_leaderboard,
    colored_pops_to_leaderboard AS
      colored_pops (player) REFERENCES player_leaderboard,
    colored_pops_to_color_stats AS
      colored_pops (player, balloon_color) REFERENCES color_stats
  )

  FACTS (
    player_leaderboard.total_score AS player_leaderboard.total_score
      WITH SYNONYMS = ('total score', 'overall score', 'total points')
      COMMENT = 'Cumulative score across all balloon pops',
    player_leaderboard.bonus_pops AS player_leaderboard.bonus_pops
      WITH SYNONYMS = ('bonus pops', 'bonuses', 'bonus count')
      COMMENT = 'Number of pops where player hit their favorite color',
    color_stats.balloon_pops AS color_stats.balloon_pops
      WITH SYNONYMS = ('pops', 'pop count', 'times popped')
      COMMENT = 'Number of times this player popped this color',
    color_stats.points_by_color AS color_stats.points_by_color
      WITH SYNONYMS = ('points by color', 'color points', 'color score')
      COMMENT = 'Total points earned from popping this color',
    color_stats.bonus_hits AS color_stats.bonus_hits
      WITH SYNONYMS = ('bonus hits', 'color bonuses')
      COMMENT = 'Number of favorite-color bonus pops for this color',
    realtime_scores.window_score AS realtime_scores.total_score
      WITH SYNONYMS = ('window score', 'live score', 'current score')
      COMMENT = 'Sum of scores within the 15-second window',
    colored_pops.window_pops AS colored_pops.balloon_pops
      COMMENT = 'Pop count for this player+color in this window',
    colored_pops.window_points AS colored_pops.points_by_color
      COMMENT = 'Points for this player+color in this window',
    colored_pops.window_bonus AS colored_pops.bonus_hits
      COMMENT = 'Bonus pops for this player+color in this window',
    color_trends.avg_score_per_pop AS color_trends.avg_score_per_pop
      WITH SYNONYMS = ('efficiency', 'points per pop', 'scoring rate', 'best value')
      COMMENT = 'Average points earned per pop of this color in this window',
    color_trends.total_pops AS color_trends.total_pops
      WITH SYNONYMS = ('volume', 'popularity', 'total pops')
      COMMENT = 'Total pops of this color in this window'
  )

  DIMENSIONS (
    player_leaderboard.player_name AS player_leaderboard.player
      WITH SYNONYMS = ('player', 'gamer', 'username', 'who')
      COMMENT = 'Unique player identifier',
    player_leaderboard.last_active AS player_leaderboard.last_event_ts
      WITH SYNONYMS = ('last active', 'last seen', 'last played')
      COMMENT = 'Timestamp of the most recent game event for this player',
    color_stats.cs_player AS color_stats.player
      COMMENT = 'Player identifier in color stats',
    color_stats.color AS color_stats.balloon_color
      WITH SYNONYMS = ('balloon color', 'color', 'balloon type')
      COMMENT = 'Color of the balloon (red, blue, green, yellow, etc.)',
    color_stats.cs_last_event AS color_stats.last_event_ts
      COMMENT = 'Most recent pop of this color by this player',
    realtime_scores.rs_player AS realtime_scores.player
      COMMENT = 'Player identifier in realtime scores',
    realtime_scores.window_start AS realtime_scores.window_start
      WITH SYNONYMS = ('start time', 'window start')
      COMMENT = 'Start of the 15-second time window',
    realtime_scores.window_end AS realtime_scores.window_end
      WITH SYNONYMS = ('end time', 'window end')
      COMMENT = 'End of the 15-second time window',
    colored_pops.cp_player AS colored_pops.player
      COMMENT = 'Player identifier in colored pops',
    colored_pops.cp_color AS colored_pops.balloon_color
      COMMENT = 'Balloon color in the detailed window view',
    colored_pops.cp_window_start AS colored_pops.window_start
      COMMENT = 'Start of the time window',
    colored_pops.cp_window_end AS colored_pops.window_end
      COMMENT = 'End of the time window',
    color_trends.ct_color AS color_trends.balloon_color
      WITH SYNONYMS = ('trending color', 'color trend')
      COMMENT = 'Balloon color for performance trend analysis',
    color_trends.ct_window_start AS color_trends.window_start
      COMMENT = 'Start of the trend analysis window',
    color_trends.ct_window_end AS color_trends.window_end
      COMMENT = 'End of the trend analysis window'
  )

  METRICS (
    player_leaderboard.m_total_score AS SUM(player_leaderboard.total_score)
      WITH SYNONYMS = ('total points', 'combined score')
      COMMENT = 'Total cumulative score across all players',
    player_leaderboard.m_total_bonus_pops AS SUM(player_leaderboard.bonus_pops)
      WITH SYNONYMS = ('bonus total', 'all bonuses')
      COMMENT = 'Total bonus pops across all players',
    player_leaderboard.m_player_count AS COUNT(player_leaderboard.player)
      WITH SYNONYMS = ('number of players', 'how many players')
      COMMENT = 'Count of players on the leaderboard',
    color_stats.m_total_pops_by_color AS SUM(color_stats.balloon_pops)
      WITH SYNONYMS = ('total balloon pops', 'all pops')
      COMMENT = 'Total balloon pops aggregated across players for a given color',
    color_stats.m_total_points_by_color AS SUM(color_stats.points_by_color)
      WITH SYNONYMS = ('color points total', 'total color points')
      COMMENT = 'Total points aggregated across players for a given color',
    color_stats.m_avg_points_per_pop AS AVG(color_stats.points_by_color / NULLIF(color_stats.balloon_pops, 0))
      WITH SYNONYMS = ('efficiency', 'scoring rate', 'points per pop')
      COMMENT = 'Average points per pop across colors',
    realtime_scores.m_max_window_score AS MAX(realtime_scores.window_score)
      WITH SYNONYMS = ('best window', 'peak score', 'hottest moment')
      COMMENT = 'Highest score in any single 15-second window',
    realtime_scores.m_avg_window_score AS AVG(realtime_scores.window_score)
      WITH SYNONYMS = ('average window score', 'typical window')
      COMMENT = 'Average score per 15-second window',
    color_trends.m_avg_efficiency AS AVG(color_trends.avg_score_per_pop)
      WITH SYNONYMS = ('trend efficiency', 'color efficiency')
      COMMENT = 'Weighted average score per pop across time windows',
    color_trends.m_total_pops AS SUM(color_trends.total_pops)
      WITH SYNONYMS = ('color popularity', 'total color pops')
      COMMENT = 'Total balloon pops across all colors and time windows'
  )

  COMMENT = 'AI-ready semantic layer over balloon game silver Dynamic Iceberg Tables'

  AI_SQL_GENERATION 'This is a balloon popping game. Players pop colored balloons to earn points. Some pops are bonus pops worth extra. The leaderboard has overall rankings by total_score. Color stats show which colors each player pops most and points per color. Realtime scores show 15-second windows of activity. Color trends show which balloon colors give the best points-per-pop over time. When asked about the top player, use the leaderboard total_score. When asked which color scores best, use color_trends avg_score_per_pop. When asked who is hot right now, use realtime_scores with the most recent window_start.';

---

## Step 4: Verify the Semantic View

Confirm the view was created and inspect its structure.

In [ ]:
%%sql -r show_views
SHOW SEMANTIC VIEWS IN SCHEMA {{DB_NAME}}.{{SCHEMA_NAME}};

In [ ]:
%%sql -r desc_view
DESC SEMANTIC VIEW {{DB_NAME}}.{{SCHEMA_NAME}}.{{SEMANTIC_VIEW_NAME}};

---

## Step 5: (Optional) Set Up Email Tool

If you want the Intelligence agent to send query results via email, create the notification integration and stored procedure below. Skip this step if you only want interactive querying.

> **Requirement:** Your Snowflake user must have a verified email address for delivery to work.

In [ ]:
%%sql -r create_email_tool
-- Notification integration for email delivery
CREATE OR REPLACE NOTIFICATION INTEGRATION email_integration
  TYPE = EMAIL
  ENABLED = TRUE
  DEFAULT_SUBJECT = 'Balloon Game Analytics';

-- Stored procedure that the agent calls to send emails
CREATE OR REPLACE PROCEDURE {{DB_NAME}}.{{SCHEMA_NAME}}.send_email(
    recipient_email VARCHAR,
    subject VARCHAR,
    body VARCHAR
)
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    CALL SYSTEM$SEND_EMAIL(
        'email_integration',
        :recipient_email,
        :subject,
        :body,
        'text/html'
    );
    RETURN 'Email sent successfully to ' || :recipient_email;
END;

---

## Step 6: Create the Snowflake Intelligence Agent

Now configure an agent in the Snowsight UI that uses your Semantic View for natural-language querying.

**1. Navigate to the Agent admin page:**
- In Snowsight, go to **AI & ML → Agents**
- Confirm your role is set to **ACCOUNTADMIN** (top-right role selector)

**2. Create a new agent:**
- Click **+ Create agent**
- **Agent object name:** `balloon_game_agent`
- **Display name:** `Balloon Game Analytics`
- **Description:** *"Ask questions about balloon game player scores, color stats, and performance trends from the silver lakehouse tables."*
- Click **Create agent**

**3. Add the Cortex Analyst tool (Semantic View):**
- Select the **Tools** tab
- Find **Cortex Analyst** and click **+ Add**
- Choose **Semantic View** (not "Semantic model file")
- Select database: `{{DB_NAME}}`, schema: `{{SCHEMA_NAME}}`, view: `{{SEMANTIC_VIEW_NAME}}`
- For **Description**, click **Generate with Cortex** to auto-generate — or write: *"Queries structured balloon game data including player leaderboards, color stats, real-time scores, and performance trends. Use for any question about players, scores, colors, or time-based patterns."*
- Set the **Warehouse** to your lab warehouse

**4. (If you ran Step 5) Add the Email tool:**
- In the **Tools** tab, find **Custom Tools** and click **+ Add**
- Select database: `{{DB_NAME}}`, schema: `{{SCHEMA_NAME}}`, procedure: `send_email`
- Configure parameter descriptions (these guide the agent on how to use the tool):
  - **recipient_email:** *"If the email is not provided, send it to the current user's email address."*
  - **subject:** *"If subject is not provided, use 'Balloon Game Analytics'."*
  - **body:** *"If body is not provided, summarize the last question and use that as content for the email."*

**5. Add sample questions:**
- Select the **Voice** tab (or **Instructions** tab depending on your Snowsight version)
- Under **Sample questions**, add:
  - *"Who are the top 5 players by total score?"*
  - *"Which balloon color gives the best average points per pop?"*
  - *"Show me score trends over the last few time windows"*
  - *"How many total bonus pops have all players earned?"*
  - *"Email me a summary of the top 3 players"*

**6. Set orchestration instructions:**
- In the **Instructions** section, add the following orchestration instruction:
  - *"Whenever you can answer visually with a chart, always choose to generate a chart even if the user didn't specify to."*

**7. Save the agent** — Click **Save** in the top-right corner. The agent is now live.

---

## Step 7: Try It!

Open your agent and ask questions in natural language.

**Access the agent:**
- In Snowsight: **AI & ML → Snowflake Intelligence** → select `Balloon Game Analytics` from the agent picker
- Or go to [ai.snowflake.com](https://ai.snowflake.com) and select the agent

**Example questions to try:**

> Who are the top 5 players by total score?

> What's the most popular balloon color across all players?

> Which color gives the best average points per pop?

> Show me how player scores trend over time windows

> Which players have the most bonus pops as a percentage of total pops?

> Email me the top 3 players leaderboard

The agent routes your question to Cortex Analyst, which reads the Semantic View's metadata to generate accurate SQL against your silver Dynamic Iceberg Tables.

---

## What Just Happened?

You made your silver lakehouse data **AI-ready** using Snowflake Intelligence:

| Component | What it does |
|---|---|
| **Semantic View** | Maps business concepts (players, scores, colors) to physical tables and columns — the "brain" of the agent |
| **Cortex Analyst** | Converts natural-language questions into SQL grounded in the Semantic View |
| **Intelligence Agent** | Orchestrates tools (Analyst, Email) and provides a chat UI for business users |
| **Email Tool** | (Optional) Sends query results to users via email |

Key takeaways:
- **No code** — business users ask questions in plain English
- **Governed** — the agent respects RBAC; users only see data their role can access
- **Grounded** — answers come from SQL against real data, not hallucinated by an LLM
- **Same data** — the same Iceberg tables power dashboards (SiS), cross-engine queries (DuckDB), **and** AI analytics (Intelligence)

**Next up:** Build a Streamlit in Snowflake dashboard to visualize these silver tables — open the **SiS Dashboard** chapter in the quickstart guide.